# Comprehensive Three-Approach Model Performance Comparison
## Best Single Model | Basin-Specific 3-Model Ensemble | Equal-Weight 6-Model Ensemble
### Reference Period: 1995–2014 | SSP2-4.5

**Purpose:**  

**Analyses performed:**

1. **Overall metric distributions** — box plots equivalent (median, IQR, range)    for all five metrics across 12 basins per approach  
2. **Basin-by-basin winner identification** — which approach performs best on    each metric for each basin  
3. **Spatial pattern of PBIAS amplification** — does bias increase with    ensemble size concentrate in arid/semi-arid eastern basins?  
4. **Aridity-stratified analysis** — performance by climate regime      
5. **Composite approach scoring** — which approach wins the most metrics    per basin; overall recommendation per basin and nationally  


**Outputs saved to:**  
`C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\validation\comparison\`

## 1. Import Libraries

In [1]:
import os
import warnings
import numpy as np
import pandas as pd
from pathlib import Path

warnings.filterwarnings("ignore")

print("Libraries loaded.")
print(f"  numpy  : {np.__version__}")
print(f"  pandas : {pd.__version__}")


Libraries loaded.
  numpy  : 2.1.3
  pandas : 2.2.3


## 2. Configuration — Input Paths

In [2]:
BASE = Path(r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\validation")

# ── Input CSVs from NB03 (Best single model) ──────────────────────────────────
SINGLE_DIR         = BASE / "single.model"
SINGLE_BASIN_CSV   = SINGLE_DIR / "basin_level_metrics.csv"
SINGLE_STN_CSV     = SINGLE_DIR / "station_level_metrics.csv"
SINGLE_TABLE3_CSV  = SINGLE_DIR / "table3_best_model_per_basin.csv"
SINGLE_RANKING_CSV = SINGLE_DIR / "basin_model_rankings.csv"

# ── Input CSVs from NB07 (3-model basin-specific ensemble) ────────────────────
ENS3_DIR       = BASE / "ensemble.3model"
ENS3_BASIN_CSV = ENS3_DIR / "basin_level_metrics_ensemble.csv"
ENS3_STN_CSV   = ENS3_DIR / "station_level_metrics_ensemble.csv"

# ── Input CSVs from NB10 (6-model equal-weight ensemble) ─────────────────────
ENS6_DIR       = BASE / "ensemble.6model"
ENS6_BASIN_CSV = ENS6_DIR / "basin_level_metrics_6ens.csv"
ENS6_STN_CSV   = ENS6_DIR / "station_level_metrics_6ens.csv"

# ── Observed LTMM for aridity classification ──────────────────────────────────
OBS_LTMM_CSV = Path(
    r"C:\Users\ASUS\Desktop\new.work.for.rainfall\comments"
    r"\stations data\long_term_monthly_mean\obs_ltmm_all_stations.csv"
)

STATION_MAPPING = Path(
    r"D:\RICAAR\Pr.New.Stations.Selection\OBSERVATIONS"
    r"\Station_Basin_Mapping.xlsx"
)

# ── Output ────────────────────────────────────────────────────────────────────
OUTPUT_ROOT = BASE / "comparison"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

APPROACH_LABELS = {
    "single" : "Best Single Model",
    "ens3"   : "3-Model Ensemble",
    "ens6"   : "6-Model Ensemble",
}

METRICS        = ["r", "NSE", "RMSE", "MAE", "PBIAS"]
ERROR_METRICS  = ["RMSE", "MAE"]          # lower is better
SKILL_METRICS  = ["r", "NSE"]             # higher is better
# PBIAS treated separately (absolute value, lower is better)

print("Configuration loaded.")
print(f"  Output : {OUTPUT_ROOT}")


Configuration loaded.
  Output : C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\validation\comparison


## 3. Load All Basin-Level Metrics

Basin-level metrics from all three approaches are loaded and merged into a single long-format DataFrame for unified analysis.

In [3]:
# ── Load raw inputs ───────────────────────────────────────────────────────────
single_b_raw = pd.read_csv(SINGLE_BASIN_CSV)   # 6 rows per basin (one per model)
ens3_b       = pd.read_csv(ENS3_BASIN_CSV)     # 1 row per basin
ens6_b       = pd.read_csv(ENS6_BASIN_CSV)     # 1 row per basin
table3       = pd.read_csv(SINGLE_TABLE3_CSV)  # best model per basin

# ── Reduce single_b to best model only (1 row per basin) ─────────────────────
# table3 has the best model name for each basin; join it to filter single_b_raw
table3_clean = table3[~table3["Basin"].isin(["Mean", "Std Dev"])][
    ["Basin", "Best_Model"]
].copy()

single_b = single_b_raw.merge(
    table3_clean, left_on=["Basin", "Model"], right_on=["Basin", "Best_Model"],
    how="inner"
).drop(columns=["Best_Model"])

# Verify: exactly 1 row per basin
assert single_b.groupby("Basin").size().max() == 1, "single_b still has duplicates!"
print(f"single_b reduced to best model: {len(single_b)} rows (1 per basin)")

# ── Standardise column names ──────────────────────────────────────────────────
# All three DataFrames need: Basin, r, NSE, PBIAS, RMSE, MAE, Approach
# single_b has 'Model', ens3_b and ens6_b have 'Model' with ensemble label

single_b["Approach"]       = "single"
ens3_b["Approach"]         = "ens3"
ens6_b["Approach"]         = "ens6"

# Harmonise to common columns
KEEP = ["Basin", "r", "NSE", "PBIAS", "RMSE", "MAE", "Approach"]

def keep_cols(df, label):
    df = df.copy()
    missing = [c for c in KEEP if c not in df.columns]
    if missing:
        raise ValueError(f"{label} missing columns: {missing}")
    return df[KEEP]

single_b_clean = keep_cols(single_b, "single_b")
ens3_b_clean   = keep_cols(ens3_b,   "ens3_b")
ens6_b_clean   = keep_cols(ens6_b,   "ens6_b")

# ── Combine — now exactly 1 row per basin per approach ───────────────────────
all_basin = pd.concat([single_b_clean, ens3_b_clean, ens6_b_clean],
                      ignore_index=True)
all_basin["Approach_Label"] = all_basin["Approach"].map(APPROACH_LABELS)
all_basin["absPBIAS"]       = all_basin["PBIAS"].abs()

# ── Verify structure ──────────────────────────────────────────────────────────
check = all_basin.groupby(["Basin", "Approach"]).size()
assert check.max() == 1, f"Duplicates found!\n{check[check > 1]}"
print(f"all_basin: {len(all_basin)} rows | {all_basin['Basin'].nunique()} basins x 3 approaches")

# ── Load station-level data ───────────────────────────────────────────────────
single_s = pd.read_csv(SINGLE_STN_CSV)
ens3_s   = pd.read_csv(ENS3_STN_CSV)
ens6_s   = pd.read_csv(ENS6_STN_CSV)

# Reduce single station data to best model only per basin
single_s = single_s.merge(
    table3_clean, left_on=["Basin", "Model"], right_on=["Basin", "Best_Model"],
    how="inner"
).drop(columns=["Best_Model"])

single_s["Approach"] = "single"
ens3_s["Approach"]   = "ens3"
ens6_s["Approach"]   = "ens6"

all_stn = pd.concat([single_s, ens3_s, ens6_s], ignore_index=True)
all_stn["Approach_Label"] = all_stn["Approach"].map(APPROACH_LABELS)
all_stn["absPBIAS"]       = all_stn["PBIAS"].abs()

# ── Basin order from station mapping ─────────────────────────────────────────
stations = pd.read_excel(STATION_MAPPING)
stations["Basin"] = stations["Basin"].astype(str).str.strip()
basin_order = list(dict.fromkeys(stations["Basin"].tolist()))
basin_order = [b for b in basin_order if b in all_basin["Basin"].unique()]

print(f"all_stn   : {len(all_stn)} rows")
print(f"Basins    : {len(basin_order)}")
print()
print("Approaches and row counts in all_basin:")
print(all_basin.groupby(["Approach","Approach_Label"]).size().to_string())


single_b reduced to best model: 12 rows (1 per basin)
all_basin: 36 rows | 12 basins x 3 approaches
all_stn   : 147 rows
Basins    : 12

Approaches and row counts in all_basin:
Approach  Approach_Label   
ens3      3-Model Ensemble     12
ens6      6-Model Ensemble     12
single    Best Single Model    12


## 4. Overall Metric Distribution Across All Basins

**Interpretation guide:**
- **r, NSE** — higher median = better temporal skill
- **RMSE, MAE** — lower median = smaller absolute errors  
- **|PBIAS|** — lower median = less systematic bias

In [4]:
dist_rows = []
for approach in ["single","ens3","ens6"]:
    sub = all_basin[all_basin["Approach"] == approach]
    for metric in ["r","NSE","RMSE","MAE","absPBIAS"]:
        vals = pd.to_numeric(sub[metric], errors="coerce").dropna()
        dist_rows.append({
            "Approach"  : APPROACH_LABELS[approach],
            "Metric"    : metric,
            "Median"    : round(vals.median(), 4),
            "Mean"      : round(vals.mean(),   4),
            "Std"       : round(vals.std(),    4),
            "Q25"       : round(vals.quantile(0.25), 4),
            "Q75"       : round(vals.quantile(0.75), 4),
            "Min"       : round(vals.min(),    4),
            "Max"       : round(vals.max(),    4),
            "N_Basins"  : len(vals),
        })

dist_df = pd.DataFrame(dist_rows)
dist_df.to_csv(OUTPUT_ROOT / "metric_distribution_summary.csv", index=False)

print("OVERALL METRIC DISTRIBUTION — BASIN-LEVEL (n=12 basins)")
print("=" * 90)

for metric in ["r","NSE","RMSE","MAE","absPBIAS"]:
    print(f" {metric}:")
    sub = dist_df[dist_df["Metric"] == metric][
        ["Approach","Median","Mean","Std","Q25","Q75"]
    ]
    print(sub.to_string(index=False))

print()
print("Saved: metric_distribution_summary.csv")


OVERALL METRIC DISTRIBUTION — BASIN-LEVEL (n=12 basins)
 r:
         Approach  Median   Mean    Std    Q25    Q75
Best Single Model  0.9702 0.9625 0.0283 0.9504 0.9839
 3-Model Ensemble  0.9747 0.9646 0.0285 0.9542 0.9851
 6-Model Ensemble  0.9659 0.9512 0.0384 0.9320 0.9772
 NSE:
         Approach  Median   Mean    Std    Q25    Q75
Best Single Model  0.7042 0.5811 0.4998 0.5662 0.8318
 3-Model Ensemble  0.5654 0.4693 0.5031 0.3185 0.8230
 6-Model Ensemble  0.5306 0.3304 0.7018 0.0330 0.8100
 RMSE:
         Approach  Median    Mean    Std    Q25     Q75
Best Single Model  8.5058  8.7054 4.0156 5.7866 11.3215
 3-Model Ensemble  8.5733  9.6752 5.4705 5.8301 11.4142
 6-Model Ensemble  8.8711 10.3649 5.3671 6.8282 12.3067
 MAE:
         Approach  Median   Mean    Std    Q25    Q75
Best Single Model  5.1722 5.4528 2.5736 3.8500 7.1266
 3-Model Ensemble  5.0094 6.1294 3.6548 4.0693 7.2607
 6-Model Ensemble  5.1930 6.5248 3.5563 4.4900 7.7052
 absPBIAS:
         Approach  Median    Mean     

## 5. Basin-by-Basin Winner Identification

For each basin and each metric, the best-performing approach is identified. The winner count per approach is then summarised nationally, directly answering: *which approach wins most often and on which metrics?*

**Winning criteria:**
- r, NSE → highest value wins  
- RMSE, MAE, |PBIAS| → lowest value wins

In [5]:
winner_rows = []

metric_better = {
    "r"       : "max",
    "NSE"     : "max",
    "RMSE"    : "min",
    "MAE"     : "min",
    "absPBIAS": "min",
}

for basin in basin_order:
    sub = all_basin[all_basin["Basin"] == basin].set_index("Approach")
    row = {"Basin": basin}

    for metric, direction in metric_better.items():
        # all_basin now has exactly 1 row per (basin, approach) so
        # .loc[appr, metric] always returns a scalar
        appr_vals = {}
        for appr in ["single", "ens3", "ens6"]:
            if appr in sub.index:
                v = pd.to_numeric(sub.loc[appr, metric], errors="coerce")
                appr_vals[appr] = float(v) if not np.isnan(v) else np.nan
            else:
                appr_vals[appr] = np.nan

        valid = {k: v for k, v in appr_vals.items() if not np.isnan(v)}

        if not valid:
            row[f"Winner_{metric}"]       = "N/A"
            row[f"Winner_{metric}_label"] = "N/A"
        else:
            winner_key = min(valid, key=valid.get) if direction == "min"                          else max(valid, key=valid.get)
            row[f"Winner_{metric}"]       = winner_key
            row[f"Winner_{metric}_label"] = APPROACH_LABELS[winner_key]

        for appr in ["single", "ens3", "ens6"]:
            v = appr_vals.get(appr, np.nan)
            row[f"{appr}_{metric}"] = round(v, 4) if not np.isnan(v) else np.nan

    # Count metric wins per approach for this basin
    win_counts = {appr: 0 for appr in ["single", "ens3", "ens6"]}
    for m in metric_better:
        w = row.get(f"Winner_{m}", "N/A")
        if w in win_counts:
            win_counts[w] += 1

    best_approach = max(win_counts, key=win_counts.get)
    row["N_wins_single"] = win_counts["single"]
    row["N_wins_ens3"]   = win_counts["ens3"]
    row["N_wins_ens6"]   = win_counts["ens6"]
    row["Basin_Winner"]  = APPROACH_LABELS[best_approach]
    winner_rows.append(row)

winner_df = pd.DataFrame(winner_rows)
winner_df.to_csv(OUTPUT_ROOT / "basin_winner_by_metric.csv", index=False)

print("BASIN-BY-BASIN WINNER SUMMARY")
print("=" * 80)
disp_cols = ["Basin", "Winner_r_label", "Winner_NSE_label",
             "Winner_RMSE_label", "Winner_MAE_label", "Winner_absPBIAS_label",
             "Basin_Winner"]
print(winner_df[disp_cols].rename(columns={
    "Winner_r_label"       : "r",
    "Winner_NSE_label"     : "NSE",
    "Winner_RMSE_label"    : "RMSE",
    "Winner_MAE_label"     : "MAE",
    "Winner_absPBIAS_label": "|PBIAS|",
    "Basin_Winner"         : "Overall Best",
}).to_string(index=False))

print()
print("NATIONAL WINNER COUNT (12 basins x 5 metrics = 60 metric-basin slots)")
print("-" * 60)
for appr, label in APPROACH_LABELS.items():
    total      = winner_df[f"N_wins_{appr}"].sum()
    basins_won = (winner_df["Basin_Winner"] == label).sum()
    print(f"  {label:<35} : {total:>3} metric-slots won  |  {basins_won:>2} basins overall-best")


BASIN-BY-BASIN WINNER SUMMARY
                Basin                 r               NSE              RMSE               MAE           |PBIAS|      Overall Best
              N.R.S.W  3-Model Ensemble  3-Model Ensemble Best Single Model  3-Model Ensemble  6-Model Ensemble  3-Model Ensemble
     YARMOUK (JORDAN)  3-Model Ensemble Best Single Model Best Single Model Best Single Model  6-Model Ensemble Best Single Model
JORDAN VALLY (JORDAN)  3-Model Ensemble Best Single Model Best Single Model Best Single Model  6-Model Ensemble Best Single Model
 AMMAN ZARQA (JORDAN)  3-Model Ensemble Best Single Model Best Single Model Best Single Model Best Single Model Best Single Model
              S.R.S.W Best Single Model Best Single Model Best Single Model  6-Model Ensemble  6-Model Ensemble Best Single Model
            D.S.R.S.W  3-Model Ensemble  3-Model Ensemble  3-Model Ensemble  3-Model Ensemble Best Single Model  3-Model Ensemble
                MUJIB Best Single Model Best Single Model Be

## 6. Spatial Pattern of PBIAS Amplification

Basins are classified by climate regime using observed long-term annual precipitation (from NB02 LTMM data):
- **Semi-arid ** (300 - 500 mm/yr) — northern highlands  
- **Arid** (100–300 mm/yr) — central plateau  
- **Desert** (<100 mm/yr) — eastern desert and southern basins

In [ ]:
# ── Compute observed annual mean per basin from LTMM ─────────────────────────
obs_ltmm = pd.read_csv(OBS_LTMM_CSV)
obs_ltmm["Station_ID"] = obs_ltmm["Station_ID"].astype(str).str.strip()

obs_annual = (
    obs_ltmm.groupby("Basin")["LTMM_mm_month"]
    .sum()       # sum of 12 monthly means = annual mean (mm/yr)
    .reset_index()
    .rename(columns={"LTMM_mm_month": "Obs_Annual_mm"})
)
obs_annual["Obs_Annual_mm"] = obs_annual["Obs_Annual_mm"].round(1)

def aridity_class(ann_mm):
    if ann_mm >= 300:
        return "Semi-arid (>300 mm/yr)"
    elif ann_mm >= 100:
        return "Semi-arid (100-300 mm/yr)"
    else:
        return "Arid (<100 mm/yr)"

obs_annual["Aridity"] = obs_annual["Obs_Annual_mm"].apply(aridity_class)

# ── PBIAS per approach per basin ──────────────────────────────────────────────
pbias_rows = []
for basin in basin_order:
    sub = all_basin[all_basin["Basin"] == basin]
    row = {"Basin": basin}
    for appr in ["single","ens3","ens6"]:
        val = sub[sub["Approach"] == appr]["absPBIAS"].values
        row[f"absPBIAS_{appr}"] = round(float(val[0]), 2) if len(val) > 0 else np.nan
    ann = obs_annual[obs_annual["Basin"] == basin]["Obs_Annual_mm"].values
    row["Obs_Annual_mm"] = round(float(ann[0]), 1) if len(ann) > 0 else np.nan
    row["Aridity"] = aridity_class(row["Obs_Annual_mm"]) if not np.isnan(row["Obs_Annual_mm"]) else "Unknown"
    # Amplification: 6-ens PBIAS minus single PBIAS (positive = 6-ens worse)
    row["PBIAS_amplification"] = round(
        row["absPBIAS_ens6"] - row["absPBIAS_single"], 2
    ) if not (np.isnan(row["absPBIAS_ens6"]) or np.isnan(row["absPBIAS_single"])) else np.nan
    pbias_rows.append(row)

pbias_df = pd.DataFrame(pbias_rows)
pbias_df.to_csv(OUTPUT_ROOT / "pbias_spatial_analysis.csv", index=False)

print("PBIAS SPATIAL ANALYSIS (|PBIAS|, %)")
print("=" * 90)
print(f"{'Basin':<28} {'Aridity':<25} {'Obs mm/yr':>9} "
      f"{'Single':>8} {'3-Ens':>8} {'6-Ens':>8} {'Amplification':>14}")
print("-" * 90)
for _, row in pbias_df.iterrows():
    print(f"{row['Basin']:<28} {row['Aridity']:<25} {row['Obs_Annual_mm']:>9.1f} "
          f"{row['absPBIAS_single']:>8.2f} {row['absPBIAS_ens3']:>8.2f} "
          f"{row['absPBIAS_ens6']:>8.2f} {row['PBIAS_amplification']:>+14.2f}")

print()
print("PBIAS by aridity class (median |PBIAS|):")
for aridity_label in ["Semi-arid (>300 mm/yr)", "Semi-arid (100-300 mm/yr)", "Arid (<100 mm/yr)"]:
    sub = pbias_df[pbias_df["Aridity"] == aridity_label]
    if sub.empty:
        continue
    print(f" {aridity_label}  (n={len(sub)} basins):")
    for col, label in [("absPBIAS_single","Single"), ("absPBIAS_ens3","3-Ens"), ("absPBIAS_ens6","6-Ens")]:
        vals = pd.to_numeric(sub[col], errors="coerce").dropna()
        print(f"    {label:<10} median={vals.median():.2f}%  mean={vals.mean():.2f}%")

print()
print("Saved: pbias_spatial_analysis.csv")


PBIAS SPATIAL ANALYSIS (|PBIAS|, %)
Basin                        Aridity                   Obs mm/yr   Single    3-Ens    6-Ens  Amplification
------------------------------------------------------------------------------------------
N.R.S.W                      Humid (>300 mm/yr)           1779.8     6.77     7.05     6.39          -0.38
YARMOUK (JORDAN)             Humid (>300 mm/yr)           1517.0    20.19     8.36     6.17         -14.02
JORDAN VALLY (JORDAN)        Humid (>300 mm/yr)            829.7    40.42     2.47     0.88         -39.54
AMMAN ZARQA (JORDAN)         Humid (>300 mm/yr)           1332.8    29.33    33.75    37.16          +7.83
S.R.S.W                      Humid (>300 mm/yr)           1396.8     6.50     3.16     1.43          -5.07
D.S.R.S.W                    Humid (>300 mm/yr)           1783.6     3.75     9.37     8.61          +4.86
MUJIB                        Humid (>300 mm/yr)           1033.5    22.05    28.09    31.06          +9.01
W. ARABA NORTH   

## 7. Aridity-Stratified Performance Analysis

In [7]:
# Merge aridity class into basin metrics
aridity_lookup = pbias_df.set_index("Basin")[["Obs_Annual_mm","Aridity"]].to_dict("index")

strat_rows = []
for appr in ["single","ens3","ens6"]:
    sub = all_basin[all_basin["Approach"] == appr].copy()
    sub["Obs_Annual_mm"] = sub["Basin"].map(lambda b: aridity_lookup.get(b, {}).get("Obs_Annual_mm", np.nan))
    sub["Aridity"]       = sub["Basin"].map(lambda b: aridity_lookup.get(b, {}).get("Aridity", "Unknown"))

    for aridity_label in ["Humid (>300 mm/yr)", "Semi-arid (100-300 mm/yr)", "Arid (<100 mm/yr)"]:
        grp = sub[sub["Aridity"] == aridity_label]
        if grp.empty:
            continue
        for metric in ["r","NSE","RMSE","MAE","absPBIAS"]:
            vals = pd.to_numeric(grp[metric], errors="coerce").dropna()
            strat_rows.append({
                "Approach"     : APPROACH_LABELS[appr],
                "Aridity"      : aridity_label,
                "N_Basins"     : len(grp),
                "Metric"       : metric,
                "Median"       : round(vals.median(), 4),
                "Mean"         : round(vals.mean(),   4),
            })

strat_df = pd.DataFrame(strat_rows)
strat_df.to_csv(OUTPUT_ROOT / "aridity_stratified_metrics.csv", index=False)

print("ARIDITY-STRATIFIED MEDIAN METRICS")
print("=" * 80)
for metric in ["r","NSE","RMSE","MAE","absPBIAS"]:
    print(f" Metric: {metric}  ({'higher=better' if metric in ['r','NSE'] else 'lower=better'})")
    pivot = strat_df[strat_df["Metric"] == metric].pivot_table(
        index="Aridity", columns="Approach", values="Median"
    )
    col_order = [v for v in APPROACH_LABELS.values() if v in pivot.columns]
    pivot = pivot[col_order]
    print(pivot.round(4).to_string())

print()
print("Saved: aridity_stratified_metrics.csv")


ARIDITY-STRATIFIED MEDIAN METRICS
 Metric: r  (higher=better)
Approach                   Best Single Model  3-Model Ensemble  6-Model Ensemble
Aridity                                                                         
Arid (<100 mm/yr)                     0.9321            0.9002            0.8643
Humid (>300 mm/yr)                    0.9733            0.9816            0.9731
Semi-arid (100-300 mm/yr)             0.9444            0.9455            0.9244
 Metric: NSE  (higher=better)
Approach                   Best Single Model  3-Model Ensemble  6-Model Ensemble
Aridity                                                                         
Arid (<100 mm/yr)                     0.6301            0.3599           -0.0930
Humid (>300 mm/yr)                    0.7599            0.5981            0.5770
Semi-arid (100-300 mm/yr)            -0.0017            0.0164           -0.3443
 Metric: RMSE  (lower=better)
Approach                   Best Single Model  3-Model Ensemble  6-Mo

## 8. Metric Change with Increasing Ensemble Size

A systematic view of how each metric changes as ensemble size grows from 1 (best single) → 3 (basin-specific) → 6 (equal weight). Positive Δ for RMSE/MAE/|PBIAS| = degradation; negative = improvement.  
This is the core quantitative finding of Section 3.2.

In [8]:
size_rows = []

for basin in basin_order:
    sub = all_basin[all_basin["Basin"] == basin].set_index("Approach")
    row = {"Basin": basin}

    for metric in ["r", "NSE", "RMSE", "MAE", "absPBIAS"]:
        def get_val(df, appr, col):
            if appr not in df.index:
                return np.nan
            v = pd.to_numeric(df.loc[appr, col], errors="coerce")
            return float(v) if not np.isnan(v) else np.nan

        v1 = get_val(sub, "single", metric)
        v3 = get_val(sub, "ens3",   metric)
        v6 = get_val(sub, "ens6",   metric)

        row[f"{metric}_single"] = round(v1, 4) if not np.isnan(v1) else np.nan
        row[f"{metric}_ens3"]   = round(v3, 4) if not np.isnan(v3) else np.nan
        row[f"{metric}_ens6"]   = round(v6, 4) if not np.isnan(v6) else np.nan
        row[f"{metric}_d_3v1"]  = round(v3 - v1, 4)             if not (np.isnan(v3) or np.isnan(v1)) else np.nan
        row[f"{metric}_d_6v1"]  = round(v6 - v1, 4)             if not (np.isnan(v6) or np.isnan(v1)) else np.nan

    size_rows.append(row)

size_df = pd.DataFrame(size_rows)
size_df.to_csv(OUTPUT_ROOT / "metric_change_with_ensemble_size.csv", index=False)

print("NATIONAL MEDIAN METRICS BY ENSEMBLE SIZE")
print("=" * 80)
print(f"  {'Metric':<14} {'Best Single':>12} {'3-Model Ens':>12} {'6-Model Ens':>12}"
      f"   {'Delta 3vs1':>11} {'Delta 6vs1':>11}")
print("-" * 80)
for metric in ["r", "NSE", "RMSE", "MAE", "absPBIAS"]:
    m1 = pd.to_numeric(size_df[f"{metric}_single"], errors="coerce").median()
    m3 = pd.to_numeric(size_df[f"{metric}_ens3"],   errors="coerce").median()
    m6 = pd.to_numeric(size_df[f"{metric}_ens6"],   errors="coerce").median()
    d3 = round(m3 - m1, 4)
    d6 = round(m6 - m1, 4)
    direction = "higher=better" if metric in ["r", "NSE"] else "lower=better"
    s3 = "+" if d3 > 0 else ""
    s6 = "+" if d6 > 0 else ""
    print(f"  {metric:<10} ({direction[:12]})  "
          f"{m1:>12.4f} {m3:>12.4f} {m6:>12.4f}"
          f"   {s3}{d3:>10.4f} {s6}{d6:>10.4f}")

print()
print("  RMSE/MAE/|PBIAS|: positive delta = degradation (worse than single model)")
print("  r / NSE         : positive delta = improvement")
print()
print("Saved: metric_change_with_ensemble_size.csv")


NATIONAL MEDIAN METRICS BY ENSEMBLE SIZE
  Metric          Best Single  3-Model Ens  6-Model Ens    Delta 3vs1  Delta 6vs1
--------------------------------------------------------------------------------
  r          (higher=bette)        0.9702       0.9748       0.9659   +    0.0046    -0.0043
  NSE        (higher=bette)        0.7042       0.5655       0.5306      -0.1387    -0.1736
  RMSE       (lower=better)        8.5058       8.5733       8.8711   +    0.0675 +    0.3652
  MAE        (lower=better)        5.1722       5.0094       5.1930      -0.1628 +    0.0208
  absPBIAS   (lower=better)       21.1204      13.3279      12.6743      -7.7925    -8.4461

  RMSE/MAE/|PBIAS|: positive delta = degradation (worse than single model)
  r / NSE         : positive delta = improvement

Saved: metric_change_with_ensemble_size.csv


## 9. Composite Approach Scoring — Final Recommendation per Basin

**Ranking direction:**
- r, NSE → descending (rank 1 = highest = best)  
- RMSE, MAE, |PBIAS| → ascending (rank 1 = lowest = best)

In [9]:
score_rows = []

for basin in basin_order:
    sub = all_basin[all_basin["Basin"] == basin].set_index("Approach")
    row = {"Basin": basin}
    rank_sum  = {appr: 0 for appr in ["single", "ens3", "ens6"]}
    n_metrics = 0

    for metric, direction in [("r", "max"), ("NSE", "max"),
                               ("RMSE", "min"), ("MAE", "min"), ("absPBIAS", "min")]:
        vals = {}
        for appr in ["single", "ens3", "ens6"]:
            if appr in sub.index:
                v = pd.to_numeric(sub.loc[appr, metric], errors="coerce")
                if not np.isnan(v):
                    vals[appr] = float(v)

        if len(vals) < 2:
            continue

        sorted_keys = sorted(vals, key=vals.get, reverse=(direction == "max"))
        for rank_pos, appr in enumerate(sorted_keys, 1):
            rank_sum[appr] += rank_pos
        n_metrics += 1

    if n_metrics == 0:
        continue

    for appr in ["single", "ens3", "ens6"]:
        row[f"AvgRank_{appr}"] = round(rank_sum[appr] / n_metrics, 3)

    best_appr = min(rank_sum, key=rank_sum.get)
    row["Recommended_Approach"] = APPROACH_LABELS[best_appr]
    row["Best_AvgRank"]         = round(rank_sum[best_appr] / n_metrics, 3)

    wrow = winner_df[winner_df["Basin"] == basin]
    if not wrow.empty:
        row["N_metrics_won_single"] = int(wrow["N_wins_single"].iloc[0])
        row["N_metrics_won_ens3"]   = int(wrow["N_wins_ens3"].iloc[0])
        row["N_metrics_won_ens6"]   = int(wrow["N_wins_ens6"].iloc[0])

    score_rows.append(row)

score_df = pd.DataFrame(score_rows).reset_index(drop=True)
score_df.to_csv(OUTPUT_ROOT / "basin_approach_recommendation.csv", index=False)

print("COMPOSITE APPROACH RANKING PER BASIN")
print("=" * 80)
print(f"  {'Basin':<28} {'Single AvgRank':>14} {'3-Ens AvgRank':>14}"
      f" {'6-Ens AvgRank':>14}  Recommended")
print("-" * 85)
for _, r in score_df.iterrows():
    print(f"  {r['Basin']:<28} {r['AvgRank_single']:>14.3f}"
          f" {r['AvgRank_ens3']:>14.3f} {r['AvgRank_ens6']:>14.3f}"
          f"  {r['Recommended_Approach']}")

print()
print("NATIONAL RECOMMENDATION SUMMARY:")
for label in APPROACH_LABELS.values():
    n = (score_df["Recommended_Approach"] == label).sum()
    print(f"  {label:<35} recommended for {n}/{len(score_df)} basins")


COMPOSITE APPROACH RANKING PER BASIN
  Basin                        Single AvgRank  3-Ens AvgRank  6-Ens AvgRank  Recommended
-------------------------------------------------------------------------------------
  N.R.S.W                               1.800          1.600          2.600  3-Model Ensemble
  YARMOUK (JORDAN)                      1.800          1.800          2.400  Best Single Model
  JORDAN VALLY (JORDAN)                 1.800          1.800          2.400  Best Single Model
  AMMAN ZARQA (JORDAN)                  1.200          1.800          3.000  Best Single Model
  S.R.S.W                               1.800          2.000          2.200  Best Single Model
  D.S.R.S.W                             2.400          1.400          2.200  3-Model Ensemble
  MUJIB                                 1.000          2.000          3.000  Best Single Model
  W. ARABA NORTH                        2.200          1.400          2.400  3-Model Ensemble
  HASA                         

## 10. Publication-Ready Comparison Table (Excel)

A formatted Excel workbook with three sheets:

1. **Approach Comparison** — all five metrics for all three approaches per basin,    with the best-performing approach highlighted in green per metric per basin  
2. **Recommendation** — composite ranking and final recommendation per basin  
3. **National Summary** — median metric values and winner counts across all basins

In [10]:
from openpyxl import Workbook
from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
from openpyxl.utils import get_column_letter

wb = Workbook()

# ── Shared styles ──────────────────────────────────────────────────────────────
thin     = Side(style="thin", color="AAAAAA")
BORDER   = Border(left=thin, right=thin, top=thin, bottom=thin)
CENTER   = Alignment(horizontal="center", vertical="center", wrap_text=True)
LEFT     = Alignment(horizontal="left",   vertical="center")
TITLE_F  = Font(name="Arial", bold=True, size=12, color="1F4E79")
NUM_FMT  = "0.000"

# Approach colour scheme
FILL_SINGLE  = PatternFill("solid", start_color="1F4E79")   # dark blue
FILL_ENS3    = PatternFill("solid", start_color="145A32")   # dark green
FILL_ENS6    = PatternFill("solid", start_color="784212")   # brown

BEST_SINGLE  = PatternFill("solid", start_color="BDD7EE")   # light blue
BEST_ENS3    = PatternFill("solid", start_color="C6EFCE")   # light green
BEST_ENS6    = PatternFill("solid", start_color="FCE4D6")   # light orange
NEUTRAL_FILL = PatternFill("solid", start_color="F2F2F2")
TITLE_FILL   = PatternFill("solid", start_color="D9E1F2")

best_fill_map = {
    "single": BEST_SINGLE,
    "ens3"  : BEST_ENS3,
    "ens6"  : BEST_ENS6,
}

def hdr_cell(ws, row, col, text, fill, font_color="FFFFFF", font_size=9):
    c = ws.cell(row=row, column=col, value=text)
    c.fill      = fill
    c.font      = Font(name="Arial", bold=True, color=font_color, size=font_size)
    c.alignment = CENTER
    c.border    = BORDER
    return c

def dat_cell(ws, row, col, value, fill=None, bold=False, fmt=None, align=CENTER):
    c = ws.cell(row=row, column=col, value=value)
    c.font      = Font(name="Arial", bold=bold, size=9)
    c.alignment = align
    c.border    = BORDER
    if fill:
        c.fill  = fill
    if fmt and isinstance(value, (int, float)):
        try:
            if not np.isnan(value):
                c.number_format = fmt
        except (TypeError, ValueError):
            pass
    return c

# ═══════════════════════════════════════════════════════════════════════════════
# SHEET 1 — Approach Comparison (all metrics, all basins, best highlighted)
# ═══════════════════════════════════════════════════════════════════════════════
ws1 = wb.active
ws1.title = "Approach Comparison"

# Title
n_cols_s1 = 17   # Basin + (r, NSE, RMSE, MAE, |PBIAS|) × 3 approaches + obs_annual
ws1.merge_cells(f"A1:{get_column_letter(n_cols_s1)}1")
ws1["A1"].value     = ("Three-Approach Performance Comparison — Jordan Basins "
                       "(1995-2014) | Green cell = best-performing approach for that metric")
ws1["A1"].font      = TITLE_F
ws1["A1"].alignment = CENTER
ws1["A1"].fill      = TITLE_FILL
ws1.row_dimensions[1].height = 22

# Row 2 — group headers: approach names
group_starts = [(2, 6, "Best Single Model", FILL_SINGLE),
                (7, 11, "Basin-Specific 3-Model Ensemble", FILL_ENS3),
                (12, 16, "Equal-Weight 6-Model Ensemble", FILL_ENS6)]

ws1.cell(row=2, column=1).value  = ""
ws1.cell(row=2, column=1).border = BORDER
ws1.cell(row=2, column=17).value  = ""
ws1.cell(row=2, column=17).border = BORDER

for start_col, end_col, label, fill in group_starts:
    ws1.merge_cells(start_row=2, start_column=start_col,
                    end_row=2, end_column=end_col)
    c = ws1.cell(row=2, column=start_col, value=label)
    c.fill      = fill
    c.font      = Font(name="Arial", bold=True, color="FFFFFF", size=10)
    c.alignment = CENTER
    c.border    = BORDER
ws1.row_dimensions[2].height = 18

# Row 3 — metric headers
metric_labels = ["r", "NSE", "RMSE (mm/mo)", "MAE (mm/mo)", "|PBIAS| (%)"]
hdr_cell(ws1, 3, 1, "Basin", PatternFill("solid", start_color="1F4E79"))
for appr_start, fill in [(2, FILL_SINGLE), (7, FILL_ENS3), (12, FILL_ENS6)]:
    for mi, ml in enumerate(metric_labels):
        hdr_cell(ws1, 3, appr_start + mi, ml, fill)
hdr_cell(ws1, 3, 17, "Obs Annual (mm/yr)", PatternFill("solid", start_color="5B2C6F"))
ws1.row_dimensions[3].height = 36

# Column widths
ws1.column_dimensions["A"].width = 24
for i in range(2, 17):
    ws1.column_dimensions[get_column_letter(i)].width = 10
ws1.column_dimensions[get_column_letter(17)].width = 12
ws1.freeze_panes = "B4"

# Data rows
metric_cols = ["r","NSE","RMSE","MAE","absPBIAS"]
appr_order  = ["single","ens3","ens6"]
appr_starts = [2, 7, 12]

for ri, basin in enumerate(basin_order):
    r = ri + 4
    dat_cell(ws1, r, 1, basin, bold=True, align=LEFT)

    basin_sub = all_basin[all_basin["Basin"] == basin].set_index("Approach")

    # Determine best approach per metric for highlighting
    best_appr_for_metric = {}
    for metric, direction in metric_better.items():
        vals = {}
        for appr in appr_order:
            if appr in basin_sub.index:
                v = basin_sub.loc[appr, metric]
                if isinstance(v, pd.Series):
                    v = v.iloc[0]
                v = pd.to_numeric(v, errors="coerce")
                if not (isinstance(v, float) and np.isnan(v)):
                    vals[appr] = float(v)
        if vals:
            best_appr_for_metric[metric] = (
                min(vals, key=vals.get) if direction == "min"
                else max(vals, key=vals.get)
            )

    for appr, start_col in zip(appr_order, appr_starts):
        for mi, metric in enumerate(metric_cols):
            col = start_col + mi
            if appr in basin_sub.index:
                v = basin_sub.loc[appr, metric]
                if isinstance(v, pd.Series):
                    v = v.iloc[0]
                v = pd.to_numeric(v, errors="coerce")
                val = float(v) if not (isinstance(v, float) and np.isnan(v)) else ""
            else:
                val = ""

            is_best = (best_appr_for_metric.get(metric) == appr)
            fill = best_fill_map[appr] if is_best else None
            dat_cell(ws1, r, col, val, fill=fill, fmt=NUM_FMT, bold=is_best)

    # Observed annual
    obs_val = pbias_df[pbias_df["Basin"] == basin]["Obs_Annual_mm"].values
    dat_cell(ws1, r, 17, float(obs_val[0]) if len(obs_val) > 0 else "",
             fill=PatternFill("solid", start_color="E8D5F5"), fmt="0.0")

# ═══════════════════════════════════════════════════════════════════════════════
# SHEET 2 — Recommendation per Basin
# ═══════════════════════════════════════════════════════════════════════════════
ws2 = wb.create_sheet("Basin Recommendation")

ws2.merge_cells("A1:J1")
ws2["A1"].value     = ("Composite Approach Ranking and Recommendation per Basin "
                       "| Lower Avg Rank = Better Overall Performance")
ws2["A1"].font      = TITLE_F
ws2["A1"].alignment = CENTER
ws2["A1"].fill      = TITLE_FILL
ws2.row_dimensions[1].height = 22

hdr2 = ["Basin","Obs Annual (mm/yr)","Aridity Class",
         "Single Avg Rank","3-Ens Avg Rank","6-Ens Avg Rank",
         "Single Metrics Won","3-Ens Metrics Won","6-Ens Metrics Won",
         "RECOMMENDED APPROACH"]
fills2 = [FILL_SINGLE, PatternFill("solid", start_color="5B2C6F"),
          PatternFill("solid", start_color="5B2C6F"),
          FILL_SINGLE, FILL_ENS3, FILL_ENS6,
          FILL_SINGLE, FILL_ENS3, FILL_ENS6,
          PatternFill("solid", start_color="E74C3C")]
col_widths2 = [24,12,22,12,12,12,12,12,12,22]

for ci, (h, f, w) in enumerate(zip(hdr2, fills2, col_widths2), 1):
    hdr_cell(ws2, 2, ci, h, f)
    ws2.column_dimensions[get_column_letter(ci)].width = w
ws2.row_dimensions[2].height = 36
ws2.freeze_panes = "A3"

rec_fill_map = {
    "Best Single Model"     : BEST_SINGLE,
    "3-Model Ensemble"      : BEST_ENS3,
    "6-Model Ensemble"      : BEST_ENS6,
}

for ri, (_, row) in enumerate(score_df.iterrows()):
    r = ri + 3
    basin = row["Basin"]
    obs_val = pbias_df[pbias_df["Basin"] == basin]["Obs_Annual_mm"].values
    aridity = pbias_df[pbias_df["Basin"] == basin]["Aridity"].values
    rec     = row["Recommended_Approach"]
    rfill   = rec_fill_map.get(rec, None)

    dat_cell(ws2, r, 1,  basin, bold=True, align=LEFT)
    dat_cell(ws2, r, 2,  float(obs_val[0]) if len(obs_val) > 0 else "", fmt="0.0")
    dat_cell(ws2, r, 3,  str(aridity[0]) if len(aridity) > 0 else "")
    dat_cell(ws2, r, 4,  row.get("AvgRank_single", ""), fmt=NUM_FMT)
    dat_cell(ws2, r, 5,  row.get("AvgRank_ens3",   ""), fmt=NUM_FMT)
    dat_cell(ws2, r, 6,  row.get("AvgRank_ens6",   ""), fmt=NUM_FMT)
    dat_cell(ws2, r, 7,  row.get("N_metrics_won_single", ""))
    dat_cell(ws2, r, 8,  row.get("N_metrics_won_ens3",   ""))
    dat_cell(ws2, r, 9,  row.get("N_metrics_won_ens6",   ""))
    dat_cell(ws2, r, 10, rec, fill=rfill, bold=True)

# ═══════════════════════════════════════════════════════════════════════════════
# SHEET 3 — National Summary (medians)
# ═══════════════════════════════════════════════════════════════════════════════
ws3 = wb.create_sheet("National Summary")

ws3.merge_cells("A1:G1")
ws3["A1"].value     = "National Median Performance Metrics — All Basins"
ws3["A1"].font      = TITLE_F
ws3["A1"].alignment = CENTER
ws3["A1"].fill      = TITLE_FILL
ws3.row_dimensions[1].height = 22

hdr3 = ["Approach","Median r","Median NSE","Median RMSE (mm/mo)",
         "Median MAE (mm/mo)","Median |PBIAS| (%)","Basins Where Overall Best"]
fills3 = [PatternFill("solid", start_color="1F4E79"),
          *[PatternFill("solid", start_color="2C3E50")] * 5,
          PatternFill("solid", start_color="E74C3C")]
col_widths3 = [30,12,12,14,14,16,16]
for ci, (h, f, w) in enumerate(zip(hdr3, fills3, col_widths3), 1):
    hdr_cell(ws3, 2, ci, h, f)
    ws3.column_dimensions[get_column_letter(ci)].width = w
ws3.row_dimensions[2].height = 36

appr_fills_s3 = {"single": BEST_SINGLE, "ens3": BEST_ENS3, "ens6": BEST_ENS6}

for ri, (appr, label) in enumerate(APPROACH_LABELS.items(), 3):
    sub = all_basin[all_basin["Approach"] == appr]
    fill = appr_fills_s3[appr]
    n_best = (score_df["Recommended_Approach"] == label).sum()

    dat_cell(ws3, ri, 1, label, fill=fill, bold=True, align=LEFT)
    for ci, metric in enumerate(["r","NSE","RMSE","MAE","absPBIAS"], 2):
        metric_vals = pd.to_numeric(sub[metric], errors="coerce").dropna()
        dat_cell(ws3, ri, ci, round(metric_vals.median(), 4), fill=fill, fmt=NUM_FMT)
    dat_cell(ws3, ri, 7, f"{n_best}/{len(score_df)}", fill=fill, bold=True)

# Save
xl_path = OUTPUT_ROOT / "three_approach_comparison_full.xlsx"
wb.save(xl_path)
print(f"Excel workbook saved: {xl_path}")
print()
print("Sheets:")
print("  Sheet 1 — Approach Comparison  (all metrics per basin, best cell highlighted)")
print("  Sheet 2 — Basin Recommendation (composite ranking + recommended approach)")
print("  Sheet 3 — National Summary     (median metrics across all 12 basins)")


Excel workbook saved: C:\Users\ASUS\Desktop\new.work.for.rainfall\comments\validation\comparison\three_approach_comparison_full.xlsx

Sheets:
  Sheet 1 — Approach Comparison  (all metrics per basin, best cell highlighted)
  Sheet 2 — Basin Recommendation (composite ranking + recommended approach)
  Sheet 3 — National Summary     (median metrics across all 12 basins)


## 11. Final Narrative Summary

Printed summary of key findings for manuscript reporting.

In [11]:
print("=" * 75)
print("KEY FINDINGS — THREE-APPROACH COMPARISON")
print("=" * 75)

# 1. Median RMSE progression
for metric, label, direction in [
    ("RMSE",     "RMSE (mm/month)", "lower=better"),
    ("MAE",      "MAE (mm/month)",  "lower=better"),
    ("absPBIAS", "|PBIAS| (%)",     "lower=better"),
    ("r",        "r",               "higher=better"),
    ("NSE",      "NSE",             "higher=better"),
]:
    vals = {}
    for appr in ["single","ens3","ens6"]:
        sub = all_basin[all_basin["Approach"] == appr]
        v   = pd.to_numeric(sub[metric], errors="coerce").dropna()
        vals[appr] = v.median()
    print(f" {label} medians  ({direction}):")
    for appr, lbl in APPROACH_LABELS.items():
        print(f"  {lbl:<35}: {vals[appr]:.4f}")
    # Direction of change
    delta_3v1 = vals['ens3'] - vals['single']
    delta_6v1 = vals['ens6'] - vals['single']
    if direction == "lower=better":
        chg3 = "WORSE" if delta_3v1 > 0 else "BETTER"
        chg6 = "WORSE" if delta_6v1 > 0 else "BETTER"
    else:
        chg3 = "BETTER" if delta_3v1 > 0 else "WORSE"
        chg6 = "BETTER" if delta_6v1 > 0 else "WORSE"
    print(f"  3-ens vs single: {delta_3v1:+.4f} ({chg3})")
    print(f"  6-ens vs single: {delta_6v1:+.4f} ({chg6})")

print()
print("-" * 75)
print("BASIN RECOMMENDATIONS:")
for _, row in score_df.iterrows():
    print(f"  {row['Basin']:<28} → {row['Recommended_Approach']}")

print()
print("-" * 75)
print("NATIONAL SUMMARY:")
for label in APPROACH_LABELS.values():
    n = (score_df["Recommended_Approach"] == label).sum()
    print(f"  {label:<35}: recommended for {n}/{len(score_df)} basins")

print()
print("=" * 75)
print("OUTPUT FILES")
print("=" * 75)
for root, dirs, files in os.walk(OUTPUT_ROOT):
    dirs[:] = sorted(d for d in dirs if not d.startswith("."))
    try:
        level = Path(root).relative_to(OUTPUT_ROOT).parts
    except ValueError:
        level = ()
    indent = "  " * len(level)
    print(f"{indent}📁 {Path(root).name}/")
    sub_indent = "  " * (len(level) + 1)
    for fname in sorted(files):
        fpath = Path(root) / fname
        sz = fpath.stat().st_size / 1024
        unit = "KB"
        if sz > 1024:
            sz /= 1024; unit = "MB"
        print(f"{sub_indent}📄 {fname}  ({sz:.1f} {unit})")


KEY FINDINGS — THREE-APPROACH COMPARISON
 RMSE (mm/month) medians  (lower=better):
  Best Single Model                  : 8.5058
  3-Model Ensemble                   : 8.5733
  6-Model Ensemble                   : 8.8711
  3-ens vs single: +0.0675 (WORSE)
  6-ens vs single: +0.3652 (WORSE)
 MAE (mm/month) medians  (lower=better):
  Best Single Model                  : 5.1722
  3-Model Ensemble                   : 5.0094
  6-Model Ensemble                   : 5.1930
  3-ens vs single: -0.1628 (BETTER)
  6-ens vs single: +0.0208 (WORSE)
 |PBIAS| (%) medians  (lower=better):
  Best Single Model                  : 21.1204
  3-Model Ensemble                   : 13.3279
  6-Model Ensemble                   : 12.6743
  3-ens vs single: -7.7925 (BETTER)
  6-ens vs single: -8.4461 (BETTER)
 r medians  (higher=better):
  Best Single Model                  : 0.9702
  3-Model Ensemble                   : 0.9747
  6-Model Ensemble                   : 0.9659
  3-ens vs single: +0.0046 (BETTER)
  6-e